# Transformer Model 
The simple neural model from the previous step did not outperform the TF-IDF baseline. 
In this notebook we will implement a **Transformer Model**. The goal is leverage a more advances, context-aware architecture to see if we can achieve a new level of performance.

## Initial Setup 

In [1]:
# Installation of the necessary libraries
!pip install transformers torch

### Load of the data and creation of the training and test sets.
Transformer models are powerfull and are designed work best on natural, unprocessed text. 
this models derive value from the full context of the sentence. Therefore stop words, punctuation and capitalization will kept in the data, as the provide valuable signals for the model

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

# Load of the original dataset
file_path = '../data/IMDB Dataset.csv'
df = pd.read_csv(file_path)

# Create a sample of 1000 reviews for consistency
df_sample = df.sample(n=1000, random_state=42)

# Define features (x) and target (y)
x = df_sample['review'] 
y = df_sample['sentiment']

# Split of the data into training/testing sets
x_train, x_test , y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42, stratify=y)


## Transformer Pipeline
We will deploy a pre-trained pipeline from the Hugging Face 'transformers' library. This pipeline uses a model that has already been fine-tuned for sentiment analysis.

In [3]:
from transformers import pipeline 

#load a pre-trained sentiment analysis model from Hugging Face
print('loading sentiment analysis pipeline...')
sentiment_pipeline = pipeline('sentiment-analysis')
print("Pipeline loaded")

No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f (https://huggingface.co/distilbert/distilbert-base-uncased-finetuned-sst-2-english).
Using a pipeline without specifying a model name and revision in production is not recommended.


loading sentiment analysis pipeline...


Device set to use cpu


Pipeline loaded


## Make predictions on the Test Set
The pipelines output requires minor formatting before the evaluation.  
It returns a list of dictionaries, each containing a label ('POSITIVE') and a confidence score. Our dataset labels are lowercase ('positive'). Therefore, a simple list comprehension is used to extract and convert the predicted labels to lowercase for a direct comparison.

In [4]:
# Convert the test set to a list for the pipeline
reviews_to_test = x_test.tolist()

# Run the pipeline on the 200 test reviews. 
# Need to add truncation=True to handle long reviews
print("Making predictions with the transformer model")
results = sentiment_pipeline(reviews_to_test, truncation=True)

# The pipeline outputs labels like 'POSITIVE' or 'NEGATIVE'. Dataset uses 'positive', let's convert them.
y_pred_transformer = [result['label'].lower() for result in results]
print('Predictions Complete.')



Making predictions with the transformer model
Predictions Complete.


## Evaluation of the Transformer model 

In [5]:
# The final report for the Transformer
print('\n Transformer Model Performance')
print(classification_report(y_test, y_pred_transformer))


 Transformer Model Performance
              precision    recall  f1-score   support

    negative       0.88      0.93      0.90       105
    positive       0.92      0.85      0.89        95

    accuracy                           0.90       200
   macro avg       0.90      0.89      0.89       200
weighted avg       0.90      0.90      0.89       200



# Final Project Results 

## Model Performance Comparison  
This table compares the weighted average F-1 score of the three models developed in this project.

| Model                       | F1-Score (Weighted Avg) |
| --------------------------- | ----------------------- |
| Baseline (TF-IDF + N-grams) | 0.71                    |
| Simple Neural  (Word Vectors + NN)  | 0.70                    |
| **Transformer** | **0.89** |

## Final Conclusion
The initial baseline model using TF-IDF and N-grams provided a strong starting point with a 0.71 F1-score. The more advanced model using spaCy's word vectors with a simple neural network did not yield a significant improvement, demonstrating that a strong classic model can be very competitive.

However, the **pre-trained Transformer model from Hugging Face proved to be vastly superior**, achieving a final F1-score of 0.89. This highlights the power of modern, context-aware architectures in understanding the nuances of the human language for sentiment analysis.

## Interactive Demo
Finally, I build an interactive demo of the transformer model, where every person without need of specific knowledge can write his own review and retrieve an analysis of the sentiment. For this we will use the library Gradio 

In [6]:
!pip install -q gradio


In [7]:
import gradio as gr

# This will be the function that configures the app
def predict_sentiment(text):
    """
    Takes a raw text string and returns a dictionary of the predicted label and its score
    """
    result = sentiment_pipeline(text, truncation=True)[0]
    label = result['label']
    score = result ['score']
    return {label: score}

# Creation of the gradio interface
iface = gr.Interface(fn=predict_sentiment,
                     inputs=gr.Textbox(lines=5, placeholder="Enter a movie review here..."),
                     outputs="label",
                     title="🎬 Movie Review Sentiment Analyzer",
                     description="Enter a movie review to see if the Transformer thinks it's POSITVE or NEGATIVE. This demo is powered by a DistilBERT model from Hugging Face.")

# Launch of the interface
iface.launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.
